In [2]:
!pip install mcp==1.29.0

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient  
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain.agents import create_agent


mcp_client = MultiServerMCPClient(
    {
        "playwright": {
            "transport": "stdio",
            "command": "npx",
            "args": [
                "-y",
                "@playwright/mcp@latest",
                "--browser",
                "chrome",
                "--isolated",
                "--viewport-size",
                "1280x800",
            ],
        }
    }
)
tools = await mcp_client.get_tools()

for tool in tools:
    print(tool)


UnsupportedOperation: fileno

In [ ]:
async with mcp_client.session("playwright") as session:
    tools = await load_mcp_tools(session)

    # OpenAI 도구 스키마와 호환되지 않는 도구 제외
    allowed = {
        "browser_navigate",
        "browser_snapshot",
        "browser_click",
        "browser_wait_for",
    }

    tools = [
        tool
        for tool in tools
        if tool.name in allowed
    ]

    print("사용할 도구:")
    for tool in tools:
        print("-", tool.name)

    agent = create_agent(
        model="openai:gpt-5.5",
        tools=tools,
        # response_format=QuoteScrapeResult,
        system_prompt="""
당신은 웹페이지에서 정보를 정확하게 수집하는 에이전트입니다.

반드시 Playwright MCP 도구를 사용하십시오.
페이지에 접속한 뒤 browser_snapshot으로 구조를 확인하십시오.
페이지에 실제로 표시된 정보만 수집하십시오.
""",
    )

    result = await agent.ainvoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": """
https://easyspub.co.kr/Main 에 접속하세요.

최근 나온 책들을 10권 알려주세요. 저자와 가격, 그리고 주요 소개를 함께 알려주세요. 아마 저자와 가격 같은 정보는 상세 페이지에 들어가야 알 수 있으니까 그렇게 해주세요. 
""",
                }
            ]
        }
    )

사용할 도구:
- browser_navigate
- browser_snapshot
- browser_click
- browser_wait_for


In [ ]:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================


https://easyspub.co.kr/Main 에 접속하세요.

최근 나온 책들을 10권 알려주세요. 저자와 가격, 그리고 주요 소개를 함께 알려주세요. 아마 저자와 가격 같은 정보는 상세 페이지에 들어가야 알 수 있으니까 그렇게 해주세요. 

================================== Ai Message ==================================
Tool Calls:
  browser_navigate (call_aAzQEpVcwvfFVmQ9nYIv0EUm)
 Call ID: call_aAzQEpVcwvfFVmQ9nYIv0EUm
  Args:
    url: https://easyspub.co.kr/Main
================================= Tool Message =================================
Name: browser_navigate

[{'type': 'text', 'text': "### Ran Playwright code\n```js\nawait page.goto('https://easyspub.co.kr/Main');\n```\n### Page\n- Page URL: https://easyspub.co.kr/Main\n- Page Title: 이지스퍼블리싱\n- Console: 1 errors, 1 warnings\n### Snapshot\n- [Snapshot](.playwright-mcp/page-2026-08-06T21-18-16-974Z.yml)\n### Events\n- New console entries: .playwright-mcp/console-2026-08-06T21-18-15-683Z.log#L1-L3", 'id': 'lc_e6e6f227-1d1a-4134-a29c-21cfaec2bf84'}]